# 项目：整理Netflix电影演员评分数据

## 分析目标

此数据分析的目的是，整理不同流派影视作品，比如喜剧片、动作片、科幻片中，各演员出演作品的平均IMDB评分，从而挖掘出各个流派中的高评分作品演员。

本实战项目的目的在于练习整理数据，从而得到可供下一步分析的数据。

## 简介

原始数据集记录了截止至2022年7月美国地区可观看的所有Netflix电视剧及电影数据。数据集包含两个数据表：`titles.csv`和`credits.csv`。

`titles.csv`包含电影及电视剧相关信息，包括影视作品ID、标题、类型、描述、流派、IMDB（一个国外的在线评分网站）评分，等等。`credits.csv`包含超过7万名出现在Netflix影视作品的导演及演员信息，包括名字、影视作品ID、人物名、演职员类型（导演/演员）等。

`titles.csv`每列的含义如下：
- id：影视作品ID。
- title：影视作品标题。
- show_type：作品类型，电视节目或电影。
- description：简短描述。
- release_year：发布年份。
- age_certification：适龄认证。
- runtime：每集电视剧或电影的长度。
- genres：流派类型列表。
- production_countries：出品国家列表。
- seasons：如果是电视剧，则是季数。
- imdb_id：IMDB的ID。
- imdb_score：IMDB的评分。
- imdb_votes：IMDB的投票数。
- tmdb_popularity：TMDB的流行度。
- tmdb_score：TMDB的评分。

`credits.csv`每列的含义如下：
- person_ID：演职员ID。
- id：参与的影视作品ID。
- name：姓名。
- character_name：角色姓名。
- role：演职员类型，演员或导演。

首先导入需要整理的数据，titles.csv和credits.csv，并赋值给original_titles和original_credits作为初始DataFrame

In [1]:
import pandas as pd

In [2]:
original_titles = pd.read_csv('titles.csv')
original_titles.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600


In [3]:
original_credits = pd.read_csv('credits.csv')
original_credits.head()

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR


## 评估、清洗数据

评估数据主要从数据整洁度和干净度两方面入手，整洁度需要符合“每行是一个观察值，每列是一个变量，每个单元格是一个值”的特点，而评估数据干净度则需要检查缺失值，重复数据，不一致数据，无效或错误数据等。

为了区分原始数据和清洗过的数据，我们将创建变量cleaned_titles作为original_titles的复制副本，创建cleaned_credits作为original_credits的复制副本，之后的清洗步骤均在cleaned_titles和cleaned_credits上操作。

In [4]:
cleaned_titles = original_titles.copy()
cleaned_credits = original_credits.copy() 

### 数据整洁度

In [5]:
cleaned_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,['documentation'],['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,"['drama', 'crime']",['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,"['drama', 'action', 'thriller', 'european']",['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,"['fantasy', 'action', 'comedy']",['GB'],NaN,tt0071853,8.2,534486.0,15.461,7.811
4,tm120801,The Dirty Dozen,MOVIE,12 American military prisoners in World War II...,1967,NaN,150,"['war', 'action']","['GB', 'US']",NaN,tt0061578,7.7,72662.0,20.398,7.600
5,ts22164,Monty Python's Flying Circus,SHOW,A British sketch comedy series with the shows ...,1969,TV-14,30,"['comedy', 'european']",['GB'],4.0,tt0063929,8.8,73424.0,17.617,8.306
6,tm70993,Life of Brian,MOVIE,"Brian Cohen is an average young Jewish man, bu...",1979,R,94,['comedy'],['GB'],NaN,tt0079470,8.0,395024.0,17.770,7.800
7,tm14873,Dirty Harry,MOVIE,When a madman dubbed 'Scorpio' terrorizes San ...,1971,R,102,"['thriller', 'action', 'crime']",['US'],NaN,tt0066999,7.7,155051.0,12.817,7.500
8,tm119281,Bonnie and Clyde,MOVIE,"In the 1930s, bored waitress Bonnie Parker fal...",1967,R,110,"['crime', 'drama', 'action']",['US'],NaN,tt0061418,7.7,112048.0,15.687,7.500
9,tm98978,The Blue Lagoon,MOVIE,Two small children and a ship's cook survive a...,1980,R,104,"['romance', 'action', 'drama']",['US'],NaN,tt0080453,5.8,69844.0,50.324,6.156


从cleaned_titles数据的前10行来看，genres列和production_countries列均存在包含多个变量值的行，需要对这两列拆分行。  
任意提取genres的一行观察。

In [6]:
cleaned_titles['genres'][1]

"['drama', 'crime']"

发现genres列的数据类型不是字符串列表，而是字符串,无法直接用explode方法拆分行，所以需要用python内置的eval函数转换数据类型，把列表字符串转换成列表本身。

In [7]:
cleaned_titles['genres'] = cleaned_titles['genres'].apply(lambda s:eval(s))

In [8]:
cleaned_titles['genres'][1]

['drama', 'crime']

已将genres列类型转换为字符串列表，因此可以使用explode函数将行拆分

In [9]:
cleaned_titles = cleaned_titles.explode('genres')
cleaned_titles

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,documentation,['US'],1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,drama,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,crime,['US'],NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,drama,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,action,['US'],NaN,tt0068473,7.7,107673.0,10.010,7.300
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5847,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021,NaN,90,comedy,['CO'],NaN,tt14585902,3.8,68.0,26.005,6.300
5848,tm1035612,Dad Stop Embarrassing Me - The Afterparty,MOVIE,"Jamie Foxx, David Alan Grier and more from the...",2021,PG-13,37,NaN,['US'],NaN,NaN,NaN,NaN,1.296,10.000
5849,ts271048,Mighty Little Bheem: Kite Festival,SHOW,"With winter behind them, Bheem and his townspe...",2021,NaN,7,family,[],1.0,tt13711094,7.8,18.0,2.289,10.000
5849,ts271048,Mighty Little Bheem: Kite Festival,SHOW,"With winter behind them, Bheem and his townspe...",2021,NaN,7,animation,[],1.0,tt13711094,7.8,18.0,2.289,10.000


In [10]:
cleaned_titles['production_countries'][0]

"['US']"

在production_countries列任意抽取一个变量,发现实际类型也是字符串而不是字符串列表，同样不能直接使用explode方法拆分，需要使用eval函数将列表字符串转换成列表

In [11]:
cleaned_titles['production_countries'] = cleaned_titles['production_countries'].apply(lambda s:eval(s))
cleaned_titles['production_countries'][1]

1    [US]
1    [US]
Name: production_countries, dtype: object

In [12]:
cleaned_titles['production_countries'][0]

['US']

检查后production_countries列的数据确认已转换为字符串列表，可以使用explode方法对行进行拆分。

In [13]:
cleaned_titles = cleaned_titles.explode('production_countries')

In [14]:
cleaned_titles.head(10)

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945,TV-MA,51,documentation,US,1.0,NaN,NaN,NaN,0.600,NaN
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,drama,US,NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,114,crime,US,NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,drama,US,NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,action,US,NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,thriller,US,NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972,R,109,european,US,NaN,tt0068473,7.7,107673.0,10.010,7.300
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,fantasy,GB,NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,action,GB,NaN,tt0071853,8.2,534486.0,15.461,7.811
3,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,comedy,GB,NaN,tt0071853,8.2,534486.0,15.461,7.811


至此cleaned_titles的结构性问题已处理，接下来处理cleaned_credits的结构性问题。

In [15]:
cleaned_credits.head(10)

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR
5,32267,tm84618,Peter Boyle,Wizard,ACTOR
6,519612,tm84618,Leonard Harris,Senator Charles Palantine,ACTOR
7,29068,tm84618,Diahnne Abbott,Concession Girl,ACTOR
8,519613,tm84618,Gino Ardito,Policeman at Rally,ACTOR
9,3308,tm84618,Martin Scorsese,Passenger Watching Silhouette,ACTOR


从前10行数据看，cleaned_credits符合整洁数据“每行是一个观察值，每列是一个变量，每个单元格是一个值”的特点，因此不存在结构性问题。

## 数据干净度

首先通过info方法，对数据内容有个大致了解

In [16]:
cleaned_titles.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 17818 entries, 0 to 5849
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    17818 non-null  object 
 1   title                 17817 non-null  object 
 2   type                  17818 non-null  object 
 3   description           17790 non-null  object 
 4   release_year          17818 non-null  int64  
 5   age_certification     10889 non-null  object 
 6   runtime               17818 non-null  int64  
 7   genres                17755 non-null  object 
 8   production_countries  17439 non-null  object 
 9   seasons               6224 non-null   float64
 10  imdb_id               17116 non-null  object 
 11  imdb_score            16976 non-null  float64
 12  imdb_votes            16945 non-null  float64
 13  tmdb_popularity       17663 non-null  float64
 14  tmdb_score            17241 non-null  float64
dtypes: float64(5), int64

cleaned_titles一共有17818个观察值，其中title，description，age_certification，genres，production_countries，seasons，imdb_id，imdb_score，imdb_votes，tmdb_popularity，tmdb_score列含有缺失值，需要后续处理。  

release_year表示年份，数据类型不应该是数字，应该为日期时间，需要转换数据类型。

In [17]:
cleaned_titles['release_year'] = pd.to_datetime(cleaned_titles['release_year'], format = '%Y' )

In [18]:
cleaned_titles['release_year']

0      1945-01-01
1      1976-01-01
1      1976-01-01
2      1972-01-01
2      1972-01-01
          ...    
5847   2021-01-01
5848   2021-01-01
5849   2021-01-01
5849   2021-01-01
5849   2021-01-01
Name: release_year, Length: 17818, dtype: datetime64[ns]

In [19]:
cleaned_credits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77801 entries, 0 to 77800
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   person_id  77801 non-null  int64 
 1   id         77801 non-null  object
 2   name       77801 non-null  object
 3   character  68029 non-null  object
 4   role       77801 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.0+ MB


cleaned_credits数据共有77801个观察值，其中变量character含有空缺值，需要在后续处理。  
person_id列数据类型不应为数字，应为字符串，需要转换数据类型。

In [20]:
cleaned_credits['person_id'] = cleaned_credits['person_id'].astype(str)
cleaned_credits['person_id']

0           3748
1          14658
2           7064
3           3739
4          48933
          ...   
77796     736339
77797     399499
77798     373198
77799     378132
77800    1950416
Name: person_id, Length: 77801, dtype: object

### 处理空缺数据

本次数据分析的目的是整理不同流派中各演员出演作品的平均IMDB评分，从而挖掘出各个流派中的高评分作品演员。  
因此title，description，age_certification，production_countries，seasons，imdb_id，imdb_votes，tmdb_popularity，tmdb_score列，即作品标题，描述，年龄限制，出品国家，季数，IMDB的ID，IMDB投票数，TMDB受欢迎程度，TMDB评分这些变量缺失并不影响本次数据分析的目的，因此可以保留这些变量。  
但genres和imdb_score，即流派和IMDB评分这两个变量缺失值会直接影响数据分析结果，因此需要进一步评估。

提取genres为缺失变量的观察值。

In [21]:
cleaned_titles.query('genres.isnull()')

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
212,tm255589,One Last Shot,MOVIE,"In this low-budget short film, two best buddie...",1998-01-01,NaN,30,NaN,NaN,NaN,NaN,NaN,NaN,1.890,5.2
619,tm341561,Like Twenty Impossibles,MOVIE,Occupied Palestine: A serene landscape now poc...,2003-01-01,NaN,16,NaN,PS,NaN,NaN,NaN,NaN,0.812,6.5
632,ts86241,Le Robe De Mariage Des Cieux,SHOW,It was with much difficulty that Ai Qing was a...,2004-01-01,TV-MA,63,NaN,NaN,1.0,NaN,NaN,NaN,0.600,NaN
636,tm404676,To and from New York,MOVIE,"While covering a story in New York City, a Sea...",2006-01-01,NaN,82,NaN,US,NaN,NaN,NaN,NaN,1.401,5.8
637,tm89054,Osuofia in London 2,MOVIE,Osuofia return to his Nigerian village with a ...,2004-01-01,NaN,72,NaN,XX,NaN,NaN,NaN,NaN,1.091,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5799,tm1040959,The Circle: The Afterparty,MOVIE,Stars of The Circle drop by to discuss Season ...,2021-01-01,NaN,35,NaN,US,NaN,NaN,NaN,NaN,1.882,10.0
5802,ts302434,Plastic Cup Boyz: Laughing My Mask Off!,SHOW,Comedy collective The Plastic Cup Boyz pour ou...,2021-01-01,NaN,33,NaN,NaN,1.0,NaN,NaN,NaN,0.683,NaN
5840,tm1216735,Sun of the Soil,MOVIE,"In 14th-century Mali, an ambitious young royal...",2022-01-01,NaN,26,NaN,NaN,NaN,NaN,NaN,NaN,1.179,7.0
5844,tm1074617,Bling Empire - The Afterparty,MOVIE,"The stars of ""Bling Empire"" discuss the show's...",2021-01-01,NaN,35,NaN,US,NaN,NaN,NaN,NaN,NaN,NaN


由于缺失分析所需的核心数据，因此把genre列为空的观察值删除，并查看该列为空值的个数： 

In [22]:
cleaned_titles = cleaned_titles.dropna(subset = ['genres'])
cleaned_titles['genres'].isnull().sum()

0

提取imdb_score缺失的观察值:

In [23]:
cleaned_titles.query('imdb_score.isnull()')

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
0,ts300399,Five Came Back: The Reference Films,SHOW,This collection includes 12 World War II-era p...,1945-01-01,TV-MA,51,documentation,US,1.0,NaN,NaN,NaN,0.600,NaN
75,tm132164,Bill Hicks: Sane Man,MOVIE,Sane Man was filmed before Bill recorded ‘Dang...,1989-01-01,R,80,comedy,US,NaN,NaN,NaN,NaN,3.377,7.5
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,documentation,JP,12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,family,JP,12.0,NaN,NaN,NaN,7.730,7.8
145,ts251477,My First Errand,SHOW,“Hajimete no Otsukai” (First Errand) is a Japa...,1991-01-01,TV-G,18,reality,JP,12.0,NaN,NaN,NaN,7.730,7.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5790,tm1094060,My Heroes Were Cowboys,MOVIE,Robin Wiltshire's painful childhood was rescue...,2021-01-01,PG,23,documentation,US,NaN,tt15084326,NaN,NaN,3.145,7.7
5791,tm1047429,Alan Saldaña: Locked Up,MOVIE,"Mexican comedian Alan Saldaña is back, poking ...",2021-01-01,NaN,49,comedy,NaN,NaN,NaN,NaN,NaN,6.670,6.0
5810,tm1225897,Social Man,MOVIE,Two competitive social media Influencers go he...,2021-01-01,NaN,96,comedy,NaN,NaN,tt20198164,NaN,NaN,NaN,NaN
5810,tm1225897,Social Man,MOVIE,Two competitive social media Influencers go he...,2021-01-01,NaN,96,drama,NaN,NaN,tt20198164,NaN,NaN,NaN,NaN


由于缺失分析所需的核心数据，因此把imdb_score列为空的观察值删除，并查看该列为空值的个数：

In [24]:
cleaned_titles = cleaned_titles.dropna(subset = ['imdb_score'])
cleaned_titles['imdb_score'].isnull().sum()

0

接下来评估cleaned_credits的character变量的空缺值。  

提取character值为空缺的观察值。

In [25]:
cleaned_credits.query('character.isnull()')

,person_id,id,name,character,role
36,3308,tm84618,Martin Scorsese,NaN,DIRECTOR
59,17727,tm154986,John Boorman,NaN,DIRECTOR
106,11475,tm127384,Terry Jones,NaN,DIRECTOR
107,11473,tm127384,Terry Gilliam,NaN,DIRECTOR
162,1063,tm120801,Robert Aldrich,NaN,DIRECTOR
...,...,...,...,...,...
77776,2363022,tm1097142,Mohamed El-Arkan,NaN,ACTOR
77777,1827884,tm1097142,Mohamed Bakir,NaN,DIRECTOR
77783,678884,tm1014599,Segun Arinze,NaN,ACTOR
77789,1962840,tm1014599,Seyi Babatope,NaN,DIRECTOR


由于character即角色姓名并不影响挖掘各流派高评分作品的演员，因此可以保留该变量空缺的观察值。

### 处理重复数据

根据数据变量含义及内容来看，cleaned_titles里不应该存在每个变量都相同的观察值，因此查看是否存在重复值。

In [26]:
cleaned_titles.duplicated().sum()

0

再检查cleaned_credits里是否存在重复数据。

In [27]:
cleaned_credits.duplicated().sum()

0

### 处理不一致数据

对于cleaned_titles，不一致数据可能存在genres和production_countries变量中，需要查看是否有多个不同值指代同一流派，多个不用值指代同一国家。

In [28]:
cleaned_titles['genres'].value_counts()

drama            3357
comedy           2419
thriller         1446
action           1339
romance          1080
crime            1066
documentation     981
family            769
animation         732
fantasy           727
european          679
scifi             647
horror            438
history           336
music             266
reality           226
war               221
sport             188
western            53
Name: genres, dtype: int64

经检查genres变量中不存在不一致数据，但仍然存在空字符串表示的流派，并非有效数据，因此需要删除这些行并查看删除后genres是否还有空字符串数据的行。

In [29]:
cleaned_titles = cleaned_titles.query('genres != ""')
cleaned_titles.query('genres == ""')

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score


检查production_countries列是否有不一致数据：

In [30]:
cleaned_titles['production_countries'].value_counts()

US    5648
IN    1610
GB    1068
JP    1046
FR     720
      ... 
GT       1
CU       1
LK       1
NP       1
FO       1
Name: production_countries, Length: 108, dtype: int64

Pandas只展示开头和结尾的一部分，要展示完整结果，需要把display.max_row设置为None，即取消行数上限。  
而我们只需要在调用value_counts方法时展示完整结果，因此可以结合option_context更改临时上限。

In [31]:
with pd.option_context('display.max_row',None):
    print(cleaned_titles['production_countries'].value_counts())

US         5648
IN         1610
GB         1068
JP         1046
FR          720
KR          637
ES          637
CA          608
DE          383
CN          295
MX          264
IT          224
BR          221
AU          217
TR          195
PH          192
AR          150
ID          149
BE          148
TW          133
NG          131
PL          126
ZA          103
NL          102
HK          102
CO           94
EG           93
DK           89
TH           87
SE           81
LB           70
NO           68
AE           52
IE           49
SG           47
XX           43
IL           42
RU           41
CL           35
CH           33
PS           32
BG           31
MY           30
SA           28
AT           28
IS           28
LU           27
NZ           27
PE           26
RO           25
QA           24
CZ           22
JO           19
FI           18
HU           18
UY           15
MA           15
PT           14
KH           10
KW           10
PR            9
PK            9
MT      

以上国家名称都是用两位国家代码表示，其中里面的Lebanon值未使用国家代码，而Lebanon的国家代码是LB，出现了70次，说明数据不一致，都表示统一国家，因此需要统一。

In [32]:
cleaned_titles['production_countries'] = cleaned_titles['production_countries'].replace('Lebanon','LB')
cleaned_titles.query('production_countries == "Lebanon"')

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score


以将production_countries变量中的Lebanon数值统一为LB。另外production_countries列中还存在空字符串，并非有效数据，但出品国家并不是分析所需的关键数据，因此可以保留这些空字符串的观察值。


对于cleaned_credits，不一致数据可能存在role变量中。

In [33]:
cleaned_credits['role'].value_counts()

ACTOR       73251
DIRECTOR     4550
Name: role, dtype: int64

DIRECTOR两种数据从输出结果看只有ACTOR和DIRECTOR两种数据，不存在不一致数据。但我们可以把该数据类型转换为category，以节省内存空间。

In [34]:
cleaned_credits['role'] = cleaned_credits['role'].astype('category')
cleaned_credits['role']

0           ACTOR
1           ACTOR
2           ACTOR
3           ACTOR
4           ACTOR
           ...   
77796       ACTOR
77797       ACTOR
77798       ACTOR
77799       ACTOR
77800    DIRECTOR
Name: role, Length: 77801, dtype: category
Categories (2, object): ['ACTOR', 'DIRECTOR']

### 处理无效或错误数据

In [35]:
cleaned_titles.describe()

,runtime,seasons,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
count,16970.000000,5954.000000,16970.000000,1.694100e+04,16842.000000,16515.000000
mean,80.912552,2.455492,6.514207,3.281655e+04,29.396307,6.846933
std,39.596172,2.869428,1.131095,1.141492e+05,93.178235,1.078831
min,0.000000,1.000000,1.500000,5.000000e+00,0.600000,1.000000
25%,45.000000,1.000000,5.800000,7.800000e+02,4.070000,6.200000
50%,90.000000,2.000000,6.600000,3.508000e+03,10.195000,6.900000
75%,107.000000,3.000000,7.300000,1.697800e+04,23.639000,7.500000
max,225.000000,42.000000,9.500000,2.294231e+06,2274.044000,10.000000


通过describe方法查看cleaned_titles的数值统计信息，并无脱离现实意义的数据。

cleaned_credits不包含数字变量，因此无需使用describe检查。

## 数据整理

In [36]:
cleaned_titles.head()

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,US,NaN,tt0075314,8.2,808582.0,40.965,8.179
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,crime,US,NaN,tt0075314,8.2,808582.0,40.965,8.179
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,drama,US,NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,action,US,NaN,tt0068473,7.7,107673.0,10.010,7.300
2,tm154986,Deliverance,MOVIE,Intent on seeing the Cahulawassee River before...,1972-01-01,R,109,thriller,US,NaN,tt0068473,7.7,107673.0,10.010,7.300


In [37]:
cleaned_credits.head()

,person_id,id,name,character,role
0,3748,tm84618,Robert De Niro,Travis Bickle,ACTOR
1,14658,tm84618,Jodie Foster,Iris Steensma,ACTOR
2,7064,tm84618,Albert Brooks,Tom,ACTOR
3,3739,tm84618,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,48933,tm84618,Cybill Shepherd,Betsy,ACTOR


分析目的是整理不同流派影视作品中，各演员出演作品的平均IMDB评分，从而挖掘出各个流派中的高评分作品演员。  
而title和credits两个数据中，id都表示影视作品的id，那么可以将title和credits两个数据以id为键进行合并，合并后每行就包含演职员信息的变量，从而可以同时获取genres、person_id和imdb_score的数据。

In [38]:
titles_and_credits = pd.merge(cleaned_titles, cleaned_credits, on = 'id', how = 'inner')
titles_and_credits

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,person_id,name,character,role
0,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,US,NaN,tt0075314,8.2,808582.0,40.965,8.179,3748,Robert De Niro,Travis Bickle,ACTOR
1,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,US,NaN,tt0075314,8.2,808582.0,40.965,8.179,14658,Jodie Foster,Iris Steensma,ACTOR
2,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,US,NaN,tt0075314,8.2,808582.0,40.965,8.179,7064,Albert Brooks,Tom,ACTOR
3,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,US,NaN,tt0075314,8.2,808582.0,40.965,8.179,3739,Harvey Keitel,Matthew 'Sport' Higgins,ACTOR
4,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976-01-01,R,114,drama,US,NaN,tt0075314,8.2,808582.0,40.965,8.179,48933,Cybill Shepherd,Betsy,ACTOR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
276104,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021-01-01,NaN,90,comedy,CO,NaN,tt14585902,3.8,68.0,26.005,6.300,736339,Adelaida Buscato,María Paz,ACTOR
276105,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021-01-01,NaN,90,comedy,CO,NaN,tt14585902,3.8,68.0,26.005,6.300,399499,Luz Stella Luengas,Karen Bayona,ACTOR
276106,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021-01-01,NaN,90,comedy,CO,NaN,tt14585902,3.8,68.0,26.005,6.300,373198,Inés Prieto,Fanny,ACTOR
276107,tm1059008,Lokillo,MOVIE,A controversial TV host and comedian who has b...,2021-01-01,NaN,90,comedy,CO,NaN,tt14585902,3.8,68.0,26.005,6.300,378132,Isabel Gaona,Cacica,ACTOR


而分析目的只需要挖掘不同流派中高评分作品的演员，不需要挖掘高评分作品的导演，因此需要将role中变量为DIRECTOR的行删除，只保留变量为ACTOR的行。

In [39]:
titles_and_credits = titles_and_credits.query('role == "ACTOR"')
titles_and_credits.query('role == "DIRECTOR"')

,id,title,type,description,release_year,age_certification,runtime,genres,production_countries,seasons,imdb_id,imdb_score,imdb_votes,tmdb_popularity,tmdb_score,person_id,name,character,role


接下来可以对合并后的数据按genres和person_id分组。  
这里演员是按person_id（演职员id），没有按name（演职员姓名）分组，因为姓名可能存在重名或者拼错的情况，person_id能更为准确的定位演员身份。

In [40]:
groupby_titles_and_credits = titles_and_credits.groupby(['genres','person_id'])
groupby_titles_and_credits

分组后再提取imdb_score使用mean函数计算不同流派各演员演出作品的平均imdb评分。

In [41]:
score_groupby_titles_and_credits = groupby_titles_and_credits['imdb_score'].mean()
score_groupby_titles_and_credits

genres   person_id
action   1000         6.866667
         100007       7.000000
         100013       6.400000
         100019       6.500000
         100020       6.500000
                        ...   
western  993735       6.500000
         998673       7.300000
         998674       7.300000
         998675       7.300000
         99940        4.000000
Name: imdb_score, Length: 168881, dtype: float64

可以使用reset_index函数将索引重置，得到一个规范的DataFrame。

In [42]:
score_groupby_titles_and_credits = score_groupby_titles_and_credits.reset_index()
score_groupby_titles_and_credits

,genres,person_id,imdb_score
0,action,1000,6.866667
1,action,100007,7.000000
2,action,100013,6.400000
3,action,100019,6.500000
4,action,100020,6.500000
...,...,...,...
168876,western,993735,6.500000
168877,western,998673,7.300000
168878,western,998674,7.300000
168879,western,998675,7.300000


接下来按genres分组，提取imdb_score列后使用max函数就能得到各流派演员演出作品的最高imdb平均评分。

In [43]:
generes_max_score = score_groupby_titles_and_credits.groupby('genres')['imdb_score'].max()
generes_max_score

genres
action           9.3
animation        9.3
comedy           9.2
crime            9.5
documentation    9.1
drama            9.5
european         8.9
family           9.3
fantasy          9.3
history          9.1
horror           9.0
music            8.8
reality          8.9
romance          9.2
scifi            9.3
sport            9.1
thriller         9.5
war              8.8
western          8.9
Name: imdb_score, dtype: float64

使用reset_index函数，将索引重置，得到一个规范的DataFrame。

In [44]:
generes_max_score = generes_max_score.reset_index()
generes_max_score

,genres,imdb_score
0,action,9.3
1,animation,9.3
2,comedy,9.2
3,crime,9.5
4,documentation,9.1
5,drama,9.5
6,european,8.9
7,family,9.3
8,fantasy,9.3
9,history,9.1


接下来将它与上面的score_groupby_titles_and_credits表格，以genre和imdb_score为键内进行连接，可以得到各流派最高评分对应的演员id。

In [45]:
genres_max_score_with_person_id = pd.merge(score_groupby_titles_and_credits, generes_max_score, on = ['genres', 'imdb_score'], how = 'inner')
genres_max_score_with_person_id

,genres,person_id,imdb_score
0,action,12790,9.3
1,action,1303,9.3
2,action,21033,9.3
3,action,336830,9.3
4,action,86591,9.3
...,...,...,...
131,war,826547,8.8
132,western,22311,8.9
133,western,28166,8.9
134,western,28180,8.9


接下来为以上最高评分的演员id补充姓名，可以先将cleaned_credits的person_id和name列提取出来，删除重复行。

In [46]:
person_id_and_name = cleaned_credits[['person_id','name']].drop_duplicates()
person_id_and_name

,person_id,name
0,3748,Robert De Niro
1,14658,Jodie Foster
2,7064,Albert Brooks
3,3739,Harvey Keitel
4,48933,Cybill Shepherd
...,...,...
77796,736339,Adelaida Buscato
77797,399499,Luz Stella Luengas
77798,373198,Inés Prieto
77799,378132,Isabel Gaona


接下来将genres_max_score_with_person_id和person_id_and_name通过person_id作为键连接，得到各流派最高评分对应的演员id及对应的演员姓名。

In [47]:
genres_max_score_with_person_id_and_name = pd.merge(genres_max_score_with_person_id, person_id_and_name, on = 'person_id')
genres_max_score_with_person_id_and_name

,genres,person_id,imdb_score,name
0,action,12790,9.3,Olivia Hack
1,scifi,12790,9.3,Olivia Hack
2,action,1303,9.3,Jessie Flower
3,animation,1303,9.3,Jessie Flower
4,family,1303,9.3,Jessie Flower
...,...,...,...,...
131,war,826547,8.8,Yuto Uemura
132,western,22311,8.9,Koichi Yamadera
133,western,28166,8.9,Megumi Hayashibara
134,western,28180,8.9,Unsho Ishizuka


连接后使用sort_values将各行按genres排序。

In [48]:
genres_max_score_with_person_id_and_name= genres_max_score_with_person_id_and_name.sort_values('genres')
genres_max_score_with_person_id_and_name

,genres,person_id,imdb_score,name
0,action,12790,9.3,Olivia Hack
12,action,336830,9.3,André Sogliuzzo
7,action,21033,9.3,Zach Tyler
17,action,86591,9.3,Cricket Leigh
2,action,1303,9.3,Jessie Flower
...,...,...,...,...
131,war,826547,8.8,Yuto Uemura
133,western,28166,8.9,Megumi Hayashibara
134,western,28180,8.9,Unsho Ishizuka
132,western,22311,8.9,Koichi Yamadera


将各行按照genres排序后发现索引顺序是乱的，因此使用resrt_index重置索引。

In [49]:
genres_max_score_with_person_id_and_name = genres_max_score_with_person_id_and_name.reset_index()
genres_max_score_with_person_id_and_name

,index,genres,person_id,imdb_score,name
0,0,action,12790,9.3,Olivia Hack
1,12,action,336830,9.3,André Sogliuzzo
2,7,action,21033,9.3,Zach Tyler
3,17,action,86591,9.3,Cricket Leigh
4,2,action,1303,9.3,Jessie Flower
...,...,...,...,...,...
131,131,war,826547,8.8,Yuto Uemura
132,133,western,28166,8.9,Megumi Hayashibara
133,134,western,28180,8.9,Unsho Ishizuka
134,132,western,22311,8.9,Koichi Yamadera


发现多出index列，因此使用drop方法将index列删除。

In [50]:
genres_max_score_with_person_id_and_name = genres_max_score_with_person_id_and_name.drop('index', axis =1)
genres_max_score_with_person_id_and_name

,genres,person_id,imdb_score,name
0,action,12790,9.3,Olivia Hack
1,action,336830,9.3,André Sogliuzzo
2,action,21033,9.3,Zach Tyler
3,action,86591,9.3,Cricket Leigh
4,action,1303,9.3,Jessie Flower
...,...,...,...,...
131,war,826547,8.8,Yuto Uemura
132,western,28166,8.9,Megumi Hayashibara
133,western,28180,8.9,Unsho Ishizuka
134,western,22311,8.9,Koichi Yamadera


这样就得到各流派的高评分作品演员数据了。